<a href="https://colab.research.google.com/github/GabrielJ07/ConfiguratorAgent/blob/main/Copy_of_Circuittelligence_Intake_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Circuittelligence Nervous System (CNS) - Intake System V1.0
PROPRIETARY AND CONFIDENTIAL

Sensory Input Layer: Pydantic-First Normalization, Deduplication, and PIR Routing.
"""

from pydantic import BaseModel, Field, HttpUrl, field_validator
from typing import Literal, List, Dict, Optional
from datetime import datetime, timezone
import hashlib
import uuid
import json

# ==========================================
# CONSTANTS & CONFIGURATION
# ==========================================
SEMANTIC_DEDUPE_THRESHOLD = 0.95
PIR_HOT_THRESHOLD = 0.4

# Trust Tier to Reinforcement Value mapping (to feed AEP Engine)
TRUST_TIER_WEIGHTS = {
    5: 1.0,   # Absolute Truth (e.g., SEC)
    4: 0.8,   # High Reliability
    3: 0.5,   # Moderate
    2: 0.2,   # Low/Social
    1: 0.05   # Rumor
}

# ==========================================
# PYDANTIC SCHEMAS (STRICT NORMALIZATION)
# ==========================================

class SourceRegistry(BaseModel):
    source_id: uuid.UUID = Field(default_factory=uuid.uuid4)
    name: str
    source_type: Literal["rss", "api", "scraper", "manual", "internal"]
    trust_tier: int = Field(ge=1, le=5)
    poll_interval_sec: int = Field(default=3600)

class PIRDefinition(BaseModel):
    pir_id: uuid.UUID = Field(default_factory=uuid.uuid4)
    intent: str
    keywords: List[str]
    # In a full build, this holds the reference to the Qdrant vector

class Entity(BaseModel):
    entity_type: Literal["competitor", "technology", "region", "person"]
    name: str

class CanonicalItem(BaseModel):
    """The normalized output of the Intake Layer, ready for AEP vetting."""
    item_id: uuid.UUID = Field(default_factory=uuid.uuid4)
    source_id: uuid.UUID
    url: str
    content_hash: str
    title: str
    raw_text: str
    trust_tier: int = Field(ge=1, le=5)
    entities: List[Entity] = Field(default_factory=list)
    pir_scores: Dict[uuid.UUID, float] = Field(default_factory=dict)
    storage_class: Literal["hot", "cold", "pending"] = Field(default="pending")
    observed_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

class IntakeSignal(BaseModel):
    """Raw inbound data from a collector."""
    signal_id: str # Used for idempotency
    source_id: uuid.UUID
    url: str
    title: str
    raw_text: str

class EmitterPayload(BaseModel):
    """The standard contract handed off to the CNS Core Engine."""
    payload_type: Literal["new_claim", "reinforcement_event"]
    target_id: uuid.UUID # Either the new CanonicalItem ID or the existing one being reinforced
    context_id: uuid.UUID # The source artifact ID
    reinforcement_weight: float
    content: Optional[str] = None # Present if new_claim

# ==========================================
# MOCK INFRASTRUCTURE (DB & VECTOR STORE)
# ==========================================

class MockVectorDB:
    def __init__(self):
        self.embeddings = {} # item_id -> text

    def semantic_search(self, text: str) -> tuple[Optional[uuid.UUID], float]:
        """Mock similarity search. Returns (Matched_ID, Score)."""
        # In reality, this uses Qdrant cosine similarity
        for item_id, stored_text in self.embeddings.items():
            if text.lower() == stored_text.lower():
                return item_id, 1.0
            # Mocking a partial match
            if text[:20] == stored_text[:20]:
                return item_id, 0.96
        return None, 0.0

class MockRelationalDB:
    def __init__(self):
        self.hashes = set()
        self.canonical_store = {}

# ==========================================
# INTAKE PIPELINE LOGIC
# ==========================================

class IntakePipeline:
    def __init__(self, sources: Dict[uuid.UUID, SourceRegistry], pirs: List[PIRDefinition]):
        self.sources = sources
        self.pirs = pirs
        self.db = MockRelationalDB()
        self.vector_db = MockVectorDB()
        self.emitter_queue: List[EmitterPayload] = []

    def generate_hash(self, url: str, text: str) -> str:
        payload = f"{url}|{text}".encode('utf-8')
        return hashlib.sha256(payload).hexdigest()

    def score_pir_alignment(self, text: str) -> Dict[uuid.UUID, float]:
        """Hybrid PIR scoring (Keywords + Mock Semantic)."""
        scores = {}
        for pir in self.pirs:
            # Simple mock keyword scoring
            match_count = sum(1 for kw in pir.keywords if kw.lower() in text.lower())
            score = min(1.0, match_count * 0.3)
            scores[pir.pir_id] = score
        return scores

    def process_signal(self, signal: IntakeSignal):
        print(f"\n[INTAKE] Processing Signal: '{signal.title[:30]}...'")

        # 1. Source Validation & Trust
        if signal.source_id not in self.sources:
            print("[ERROR] Unknown source_id. Dropping.")
            return
        source = self.sources[signal.source_id]

        # 2. Layer 1 Dedupe (Exact Hash)
        content_hash = self.generate_hash(signal.url, signal.raw_text)
        if content_hash in self.db.hashes:
            print("[DEDUPE L1] Exact hash match found. Idempotent drop.")
            return

        # 3. Layer 2 Dedupe (Semantic Similarity)
        matched_id, sim_score = self.vector_db.semantic_search(signal.raw_text)
        if sim_score >= SEMANTIC_DEDUPE_THRESHOLD:
            print(f"[DEDUPE L2] Semantic duplicate found (Score: {sim_score:.2f}). Emitting Reinforcement.")
            self.db.hashes.add(content_hash) # Prevent reprocessing this exact artifact

            payload = EmitterPayload(
                payload_type="reinforcement_event",
                target_id=matched_id,
                context_id=uuid.uuid4(),
                reinforcement_weight=TRUST_TIER_WEIGHTS[source.trust_tier]
            )
            self.emitter_queue.append(payload)
            return

        # 4. Canonicalization & Entity Extraction (Mocked)
        canonical = CanonicalItem(
            source_id=source.source_id,
            url=signal.url,
            content_hash=content_hash,
            title=signal.title,
            raw_text=signal.raw_text,
            trust_tier=source.trust_tier,
            entities=[Entity(entity_type="technology", name="Rust")]
        )

        # 5. PIR Alignment & Storage Routing
        pir_scores = self.score_pir_alignment(signal.raw_text)
        canonical.pir_scores = pir_scores

        max_pir_score = max(pir_scores.values()) if pir_scores else 0.0

        if max_pir_score > PIR_HOT_THRESHOLD:
            canonical.storage_class = "hot"
            print(f"[PIR ALIGNMENT] High alignment ({max_pir_score:.2f}). Routing to HOT storage & AEP Core.")

            # Emit to AEP Core Engine
            payload = EmitterPayload(
                payload_type="new_claim",
                target_id=canonical.item_id,
                context_id=uuid.uuid4(),
                reinforcement_weight=TRUST_TIER_WEIGHTS[source.trust_tier],
                content=canonical.raw_text
            )
            self.emitter_queue.append(payload)
        else:
            canonical.storage_class = "cold"
            print(f"[PIR ALIGNMENT] Low alignment ({max_pir_score:.2f}). Routing to COLD storage.")

        # 6. Commit to Data Stores
        self.db.hashes.add(content_hash)
        self.db.canonical_store[canonical.item_id] = canonical
        self.vector_db.embeddings[canonical.item_id] = canonical.raw_text

# ==========================================
# EXECUTION DEMONSTRATION
# ==========================================
if __name__ == "__main__":
    # Setup infrastructure
    source_sec = SourceRegistry(name="SEC EDGAR", source_type="api", trust_tier=5)
    source_blog = SourceRegistry(name="Tech Blog X", source_type="rss", trust_tier=2)

    pir_1 = PIRDefinition(intent="Track backend tech shifts", keywords=["rust", "golang", "backend", "migration"])

    pipeline = IntakePipeline(
        sources={source_sec.source_id: source_sec, source_blog.source_id: source_blog},
        pirs=[pir_1]
    )

    print("--- INITIATING INTAKE PIPELINE ---")

    # Signal 1: New, High Trust, High PIR Alignment
    sig1 = IntakeSignal(
        signal_id="sig-001",
        source_id=source_sec.source_id,
        url="https://sec.gov/123",
        title="Form 8-K: Competitor X",
        raw_text="Competitor X announces full backend migration to Rust."
    )
    pipeline.process_signal(sig1)

    # Signal 2: Exact Duplicate (Idempotent Drop)
    pipeline.process_signal(sig1)

    # Signal 3: Semantic Duplicate (Different phrasing, triggers Reinforcement Event)
    sig3 = IntakeSignal(
        signal_id="sig-003",
        source_id=source_blog.source_id,
        url="https://blog.x/rust-news",
        title="Competitor X switching to Rust",
        raw_text="Competitor X announces full backend migration to Rust for performance."
    )
    pipeline.process_signal(sig3)

    # Signal 4: Low PIR Alignment (Cold Storage)
    sig4 = IntakeSignal(
        signal_id="sig-004",
        source_id=source_blog.source_id,
        url="https://blog.x/lunch",
        title="Company Picnic",
        raw_text="Competitor X had a lovely company picnic today."
    )
    pipeline.process_signal(sig4)

    print("\n--- EMITTER QUEUE TO AEP CORE ---")
    for msg in pipeline.emitter_queue:
        print(msg.model_dump_json(indent=2))